<h1>MammoVLM V2 - Diagnostico Mamografico Asistido por IA</h1>
<h2>Vision Language Model con RAG y Metricas Avanzadas</h2>
<p>Clasificacion BI-RADS Multiclase (0-5) con literatura medica integrada</p>
<hr/>
<table>
<tr><td><b>Investigador:</b></td><td>Geovanny Enrique Trujillo Delgado</td></tr>
<tr><td><b>Director:</b></td><td>Noel Perez-Perez, PhD</td></tr>
<tr><td><b>Institucion:</b></td><td>Universidad San Francisco de Quito (USFQ)</td></tr>
<tr><td><b>Centro de Investigacion:</b></td><td>Por ser definido</td></tr>
</table>

<h2>Tabla de Contenidos</h2>
<ul>
<li><a href='#config'>1. Configuración del Entorno</a></li>
<li><a href='#imports'>2. Módulos del Proyecto</a></li>
<li><a href='#rag'>3. Índice RAG de Literatura Médica</a>
  <ul>
  <li>3.1 Verificación de literatura disponible</li>
  <li>3.2 Construcción del índice RAG</li>
  <li>3.3 Prueba de recuperación</li>
  </ul>
</li>
<li><a href='#area2'>4. Arquitectura MammoVLM</a>
  <ul>
  <li>4.1 Definición del modelo</li>
  <li>4.2 Verificación del forward pass</li>
  <li>4.3 Función de loss multi-tarea</li>
  </ul>
</li>
<li><a href='#entrenamiento'>5. Entrenamiento del Clasificador</a>
  <ul>
  <li>5.1 Preparación de datos &bull; 5.2 Configuración &bull; 5.3 Lanzamiento</li>
  <li>5.4 Monitoreo &bull; 5.5 Carga de resultados &bull; 5.6 Curvas de aprendizaje</li>
  <li>5.7 Evaluación en test set &bull; 5.8 Threshold tuning clínico</li>
  </ul>
</li>
<li><a href='#experimentos'>6. Historial de Experimentos</a></li>
<li><a href='#pipeline'>7. Inferencia Diagnóstica End-to-End</a>
  <ul>
  <li>7.1 Setup autónomo &bull; 7.2 Cargar el clasificador &bull; 7.3 Cargar el generador</li>
  <li>7.4 Inferencia sobre una imagen &bull; 7.5 Generación de informes por lotes</li>
  </ul>
</li>
<li><a href='#evaluacion'>8. Evaluación del Generador de Informes</a>
  <ul>
  <li>8.1 Setup autónomo &bull; 8.2 Evaluación factual del generador</li>
  </ul>
</li>
</ul>

<h2 id='config'>1. Configuración del Entorno</h2>
<p>Instalación de dependencias e inicialización de rutas del proyecto</p>

In [ ]:
## Instalar dependencias necesarias
## Ejecutar solo la primera vez o cuando se actualicen paquetes
## !pip install pymupdf4llm faiss-cpu sentence-transformers --quiet --break-system-packages

In [ ]:
import sys
import os
from pathlib import Path

## Agregar src al path para imports
PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import warnings
import logging
import json
from datetime import datetime
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## Configurar warnings y logging
warnings.filterwarnings('ignore')
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s'
)

## Estilo de plots
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('muted')

## Rutas del proyecto
SRC_DIR = PROJECT_ROOT / 'src'
DATA_DIR = PROJECT_ROOT / 'data'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
RESULTS_DIR = PROJECT_ROOT / 'results'
CHECKPOINTS_DIR = PROJECT_ROOT / 'checkpoints'

## Ruta de libros de literatura medica en el servidor H200
LITERATURE_DIR = PROJECT_ROOT / 'Libros'

## Ruta del indice RAG (se genera una vez y se reutiliza)
RAG_INDEX_DIR = DATA_DIR / 'rag_index'

## Crear directorios
for d in [DATA_DIR, OUTPUTS_DIR, RESULTS_DIR, CHECKPOINTS_DIR, RAG_INDEX_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Entorno inicializado')
print(f'  Proyecto: {PROJECT_ROOT}')
print(f'  Literatura medica: {LITERATURE_DIR}')
print(f'  Indice RAG: {RAG_INDEX_DIR}')
print(f'  Timestamp: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')

<h2 id='imports'>2. Módulos del Proyecto</h2>

In [ ]:
## Importar modulos del proyecto
from config import MammoVLMConfig
from utils import (
    setup_logging, save_json, load_json, get_timestamp,
    AverageMeter, print_separator, print_section,
)
from medical_vocabulary import BIRADSLexicon, birads_lexicon
from coherence_metrics import PredictionRecord, CoherenceValidator
from rag import (
    PDFExtractor, EmbeddingModel, CorpusIndexer,
    ReportRetriever, augment_prompt, create_rag_pipeline,
)
## Modelo (exp08): encoder Mammo-CLIP (EfficientNet-B5) + dual-head BI-RADS/densidad
from models import MammoCLIPEncoder, ClassificationHead, MammoVLM, MultiTaskLoss
from report_generator import PromptBuilder, ReportGenerator, load_llm_for_generation
## Metricas clinicas (usadas en evaluacion del test set, seccion 5.7)
from medical_metrics import (
    BIRADSClassificationMetrics, ClinicalSeverityMetrics,
    InterRaterReliability, ClinicalErrorAnalysis, MedicalMetricsReport,
)
## Carga de datos y entrenamiento
from data_loading import (
    MammoCLIPTransform, MammoDataset,
    load_vindr_records, birads_to_index, index_to_birads,
)
from train import (
    TrainingConfig, train_mammovlm,
    compute_birads_class_weights, split_train_val,
)
## Tracking de experimentos
from experiment_tracker import ExperimentTracker


## Logger del notebook
logger = setup_logging('MammoVLM', log_file=OUTPUTS_DIR / f'notebook_{get_timestamp()}.log')

## Config global
config = MammoVLMConfig()

print('Modulos importados exitosamente')
print(f'  Vocabulario BI-RADS: {len(birads_lexicon.get_all_descriptors())} descriptores')
print(f'  Categorias BI-RADS: 0-5 ({len(birads_lexicon.assessment_categories)} niveles)')


<h2 id='rag'>3. Índice RAG de Literatura Médica</h2>
<p>Extracción de texto de libros de radiología mamaria, generación de embeddings con PubMedBERT y construcción del índice FAISS para recuperación de contexto clínico.</p>

<h3>3.1 Verificación de literatura disponible</h3>

In [ ]:
## Verificar que los libros estan disponibles en el servidor
print_section('LITERATURA MEDICA DISPONIBLE', width=70)
print(f'Directorio: {LITERATURE_DIR}')
print()

if LITERATURE_DIR.exists():
    pdf_files = sorted(LITERATURE_DIR.glob('*.pdf'))
    print(f'Archivos PDF encontrados: {len(pdf_files)}')
    print()
    total_size = 0
    for i, pdf in enumerate(pdf_files, 1):
        size_mb = pdf.stat().st_size / (1024 * 1024)
        total_size += size_mb
        print(f'  {i:2d}. {pdf.name[:60]:60s} ({size_mb:6.1f} MB)')
    print()
    print(f'  Total: {total_size:.1f} MB en {len(pdf_files)} archivos')
else:
    print('ADVERTENCIA: Directorio de literatura no encontrado')
    print(f'  Ruta esperada: {LITERATURE_DIR}')
    print('  Verifica que los PDFs esten en el servidor H200')

<h3>3.2 Construcción del índice RAG</h3>
<p>Extrae texto de los PDFs, genera embeddings con PubMedBERT y construye el índice FAISS. Se ejecuta una sola vez y se guarda en caché.</p>

In [ ]:
## Construir pipeline RAG
## La primera ejecucion tarda varios minutos (extraccion + embeddings)
## Las ejecuciones siguientes cargan el indice desde cache

print_section('CONSTRUCCION DEL INDICE RAG', width=70)

if LITERATURE_DIR.exists():
    indexer, retriever = create_rag_pipeline(
        pdf_dir=str(LITERATURE_DIR),
        index_dir=str(RAG_INDEX_DIR),
        device='auto',
        force_rebuild=False,
    )
    
    ## Mostrar estadisticas del indice
    stats = indexer.get_index_stats()
    print()
    print(f'Estado: {stats["status"]}')
    print(f'Total chunks indexados: {stats["total_chunks"]}')
    print(f'Dimension embeddings: {stats["embedding_dim"]}')
    print()
    print('Chunks por fuente:')
    for source, count in sorted(stats.get('chunks_per_source', {}).items()):
        print(f'  {source[:55]:55s} {count:5d} chunks')
else:
    print('Indice RAG no construido (literatura no disponible)')
    print('El notebook continuara sin RAG')
    indexer = None
    retriever = None

<h3>3.3 Prueba de recuperación RAG</h3>

In [ ]:
## Probar recuperacion con queries de ejemplo
print_section('PRUEBA DE RECUPERACION RAG', width=70)

if retriever is not None:
    ## Query 1: Masa espiculada BI-RADS 5
    query1 = retriever.build_query_from_findings(
        birads_pred=5,
        findings=['spiculated mass', 'irregular margins'],
        density='heterogeneously dense',
    )
    results1 = retriever.retrieve(query1, top_k=3)
    
    print(f'Query 1: "{query1}"')
    print(f'Resultados: {len(results1)}')
    for i, r in enumerate(results1, 1):
        print(f'  [{i}] Score: {r["score"]:.4f} | Fuente: {r["source"]}, p.{r["page"]}')
        print(f'      {r["text"][:120]}...')
    print()
    
    ## Query 2: Calcificaciones benignas BI-RADS 2
    query2 = retriever.build_query_from_findings(
        birads_pred=2,
        findings=['round calcifications', 'scattered'],
        density='fatty',
    )
    results2 = retriever.retrieve(query2, top_k=3)
    
    print(f'Query 2: "{query2}"')
    print(f'Resultados: {len(results2)}')
    for i, r in enumerate(results2, 1):
        print(f'  [{i}] Score: {r["score"]:.4f} | Fuente: {r["source"]}, p.{r["page"]}')
        print(f'      {r["text"][:120]}...')
    print()
    
    ## Query 3: Probar augment_prompt
    base_prompt = (
        '<|im_start|>system\n'
        'Usted es un asistente de radiologia especializado en mamografia.<|im_end|>\n'
        '<|im_start|>user\n'
        '[IMG]imagen mamografica[/IMG]\n'
        'Analice esta mamografia.<|im_end|>\n'
        '<|im_start|>assistant\n'
    )
    augmented = augment_prompt(base_prompt, results1, language='es')
    print('Prompt aumentado (primeros 500 chars):')
    print(augmented[:500])
    print('...')
    print(f'Longitud original: {len(base_prompt)} chars')
    print(f'Longitud aumentado: {len(augmented)} chars')
else:
    print('RAG no disponible. Continuando sin recuperacion de literatura.')

<h2 id='area2'>4. Arquitectura MammoVLM</h2>
<p>Arquitectura dual-head con encoder Mammo-CLIP (EfficientNet-B5). Predice clasificación BI-RADS (5 clases) y densidad ACR (4 clases); el generador de informes integra el LLM Qwen2.5 con recuperación RAG.</p>

<h3>4.1 Definición del modelo</h3>

In [ ]:
## Importar arquitectura del modelo
import torch
from models import MammoVLM, MultiTaskLoss

## Detectar dispositivo
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Dispositivo: {device}')
print()

## Nombre del experimento actual
## Se usa para aislar los checkpoints de cada experimento en su propia carpeta
## y debe coincidir con el EXPERIMENT_ID que se registra en la celda 6.1
EXPERIMENT_NAME = 'exp09_asymmetric_sord_weighted'

## Ruta del checkpoint preentrenado de Mammo-CLIP (EfficientNet-B5)
## Descargado de Hugging Face (shawn24/Mammo-CLIP) y copiado a la carpeta del proyecto
MAMMOCLIP_CHECKPOINT = str(PROJECT_ROOT / 'models' / 'mammo_clip_b5.tar')

## Configuracion del encoder: numero de bloques finales a descongelar
## 0 = encoder totalmente congelado (solo se entrenan los heads)
## >0 = fine-tuning parcial del encoder (descongela los ultimos N bloques)
## IMPORTANTE: este valor debe coincidir con el usado en el entrenamiento
## (celda 5.2) para que los pesos guardados carguen correctamente
UNFREEZE_LAST_N_BLOCKS = 2

## Inicializar modelo con Mammo-CLIP (EfficientNet-B5, alta resolucion)
## El encoder esta especializado en mamografia y preentrenado en VinDr
print_section('INICIALIZANDO MammoVLM (Mammo-CLIP + Dual-Head)', width=70)

model = MammoVLM(
    checkpoint_path=MAMMOCLIP_CHECKPOINT,
    efficientnet_name='efficientnet-b5',
    num_birads_classes=5,    ## VinDr: BI-RADS 1-5 (cinco niveles)
    num_density_classes=4,   ## densidad ACR: DENSITY A-D
    freeze_encoder=True,
    unfreeze_last_n_blocks=UNFREEZE_LAST_N_BLOCKS,
    hidden_dim=256,
    dropout=0.2,
)
model = model.to(device)

print()
print(f'Parametros totales:     {model.get_total_parameters():,}')
print(f'Parametros entrenables: {model.get_trainable_parameters():,}')
if UNFREEZE_LAST_N_BLOCKS > 0:
    print(f'  (encoder con {UNFREEZE_LAST_N_BLOCKS} bloques finales descongelados)')
else:
    print(f'  (encoder congelado, solo se entrenan los heads)')

<h3>4.2 Verificación del forward pass</h3>

In [ ]:
## Probar forward pass con datos sinteticos
## El modelo recibe una sola imagen de alta resolucion (sin tiling)
## En produccion la imagen proviene del transform sobre mamografias reales de VinDr
## Se usa una resolucion reducida en la prueba sintetica para que sea rapida
dummy_images = torch.randn(2, 3, 512, 512).to(device)

outputs = model(dummy_images)
print_section('FORWARD PASS (batch=2)', width=70)
print('Entrada:')
print(f'  Imagen: {tuple(dummy_images.shape)}')
print(f'  (en produccion: {IMAGE_HEIGHT}x{IMAGE_WIDTH}, resolucion nativa de Mammo-CLIP)' if 'IMAGE_HEIGHT' in dir() else '')
print()
print('Logits por cabeza:')
for key, val in outputs.items():
    print(f'  {key:28s} shape={tuple(val.shape)}')
print()

## Prediccion (BI-RADS + densidad)
results = model.predict(dummy_images)
print('PREDICCION (imagen 0):')
print(f'  BI-RADS: indice {results[0]["birads_pred"]} '
      f'(= BI-RADS {results[0]["birads_pred"] + 1}, confianza: {results[0]["birads_confidence"]:.3f})')
print(f'  Densidad: indice {results[0]["density_pred"]} '
      f'(confianza: {results[0]["density_confidence"]:.3f})')

<h3>4.3 Función de loss multi-tarea</h3>

In [ ]:
## Configurar loss multi-tarea (BI-RADS + densidad)
loss_fn = MultiTaskLoss(
    birads_weight=1.0,      ## peso principal: clasificacion BI-RADS
    density_weight=0.5,     ## peso secundario: densidad
)

## Crear targets sinteticos para demostracion
## BI-RADS: indices 0-4 (5 clases), densidad: indices 0-3 (4 clases)
targets = {
    'birads': torch.randint(0, 5, (2,)).to(device),
    'density': torch.randint(0, 4, (2,)).to(device),
}

## Calcular loss
losses = loss_fn(outputs, targets)
print_section('LOSS MULTI-TAREA', width=70)
print(f'  Loss total:    {losses["total"].item():.4f}')
print(f'  Loss BI-RADS:  {losses["birads"].item():.4f}')
print(f'  Loss densidad: {losses["density"].item():.4f}')
print()
print('Formula: total = birads_weight * L_birads + density_weight * L_density')
print(f'         {losses["total"].item():.4f} = 1.0 * {losses["birads"].item():.4f} + 0.5 * {losses["density"].item():.4f}')

<hr/>
<h2 id='entrenamiento'>5. Entrenamiento del Clasificador</h2>
<p>Entrenamiento del clasificador MammoVLM sobre VinDr-Mammo. El encoder Mammo-CLIP se descongela parcialmente (últimos 2 bloques); se optimizan el encoder parcial y las cabezas de clasificación BI-RADS + densidad con loss ordinal SORD+QWK asimétrico.</p>
<table>
<tr><th>Dataset</th><th>Cabezas</th><th>Loss</th></tr>
<tr><td>VinDr-Mammo</td><td>BI-RADS (5 clases) + Densidad ACR (4 clases)</td><td>SORD + λ·QWK (asimétrico, β=2)</td></tr>
</table>

<h3>5.1 Preparación de datos</h3>

In [ ]:
## Construir el transform de alta resolucion para Mammo-CLIP
## A diferencia de la version anterior (multi-escala con tiling 3x3), Mammo-CLIP
## trabaja a alta resolucion nativa (1520x912), preservando microcalcificaciones
## sin necesidad de parches. CLAHE realza el contraste local.
from data_loading import MammoCLIPTransform, MammoDataset, load_vindr_records

## Resolucion nativa de Mammo-CLIP (alto x ancho)
IMAGE_HEIGHT = 1520
IMAGE_WIDTH  = 912

train_transform = MammoCLIPTransform(
    height=IMAGE_HEIGHT,
    width=IMAGE_WIDTH,
    augment=True,
    use_clahe=True,
)

## Transform de validacion (sin augmentation)
val_transform = MammoCLIPTransform(
    height=IMAGE_HEIGHT,
    width=IMAGE_WIDTH,
    augment=False,
    use_clahe=True,
)

print_section('PREPARACION DE DATOS (VinDr ALTA RESOLUCION)', width=70)
print('Procesamiento de alta resolucion configurado:')
print(f'  Resolucion: {IMAGE_HEIGHT}x{IMAGE_WIDTH} (nativa de Mammo-CLIP)')
print(f'  Sin tiling: el encoder de alta resolucion preserva el detalle fino')
print(f'  Realce de contraste: CLAHE habilitado')
print(f'  Normalizacion ImageNet: mean={MammoCLIPTransform.IMAGENET_MEAN}')
print()

## Cargar registros de VinDr para verificar conteos antes de entrenar
VINDR_ROOT = str(DATA_DIR / 'vindr-mammo')
print(f'Cargando registros de VinDr desde: {VINDR_ROOT}')
records_vindr = load_vindr_records(VINDR_ROOT)
print(f'  Total de registros VinDr: {len(records_vindr):,}')
print()

## Distribucion BI-RADS (1-5) y densidad
if len(records_vindr) > 0:
    from collections import Counter
    dist_birads = Counter(int(b) for b in records_vindr['birads'])
    print(f'  Distribucion BI-RADS (1-5): {dict(sorted(dist_birads.items()))}')
    ## Distribucion del split oficial
    if 'split' in records_vindr.columns:
        dist_split = Counter(str(s).lower() for s in records_vindr['split'])
        print(f'  Split oficial VinDr: {dict(dist_split)}')

<h3>5.2 Configuración del entrenamiento</h3>

In [ ]:
## Configuracion del entrenamiento de exp09 (una sola etapa, VinDr)
## exp09: SORD ASIMETRICO (penaliza sub-diagnostico, beta=2.0) sobre la base
## de exp08. Unico cambio respecto a exp08: la asimetria. Todo lo demas igual
## (descongelamiento 2 bloques, lambda=0.3, weight_power=0.5) para aislar el efecto.
##
## MODO PRUEBA DE HUMO: para validar que el pipeline corre de extremo a extremo
## sin esperar el entrenamiento completo, se activa PRUEBA_DE_HUMO = True.
## Esto limita a 200 muestras y 1 epoca. Para el entrenamiento real,
## cambiar PRUEBA_DE_HUMO = False.
PRUEBA_DE_HUMO = False

if PRUEBA_DE_HUMO:
    print('*** MODO PRUEBA DE HUMO ACTIVADO ***')
    print('    200 muestras, 1 epoca')
    print('    Para entrenamiento completo: cambiar PRUEBA_DE_HUMO = False')
    print()
    train_config = TrainingConfig(
        epochs          = 1,
        lr              = 1e-3,
        batch_size      = 4,         ## batch pequeno (alta resolucion usa mas VRAM)
        num_workers     = 4,
        val_fraction    = 0.15,
        random_seed     = 42,
        checkpoint_dir  = str(OUTPUTS_DIR / 'checkpoints' / f'{EXPERIMENT_NAME}_smoke'),
        test_set_dir    = str(OUTPUTS_DIR / 'test_sets'),
        device          = 'auto',
        image_height    = IMAGE_HEIGHT,
        image_width     = IMAGE_WIDTH,
        unfreeze_last_n_blocks = UNFREEZE_LAST_N_BLOCKS,  ## de la celda 4.1 (=2)
        max_samples     = 200,       ## limite para prueba de humo
        use_focal_loss  = False,
        use_oversampling = False,
        use_ordinal_loss = True,     ## SORD + lambda*QWK
        ordinal_lambda_qwk = 0.3,
        ordinal_distance_power = 1.0,
        ordinal_undergrade_beta = 2.0,  ## exp09: penaliza sub-diagnostico
        weight_power     = 0.5,      ## class weights suaves
        use_official_split = True,
    )
else:
    train_config = TrainingConfig(
        epochs          = 15,
        lr              = 1e-3,
        batch_size      = 8,         ## ajustar segun VRAM disponible en alta resolucion
        num_workers     = 4,
        weight_decay    = 1e-4,
        val_fraction    = 0.15,
        random_seed     = 42,
        checkpoint_dir  = str(OUTPUTS_DIR / 'checkpoints' / EXPERIMENT_NAME),
        test_set_dir    = str(OUTPUTS_DIR / 'test_sets'),
        device          = 'auto',
        image_height    = IMAGE_HEIGHT,
        image_width     = IMAGE_WIDTH,
        unfreeze_last_n_blocks = UNFREEZE_LAST_N_BLOCKS,  ## de la celda 4.1 (=2)
        max_samples     = None,      ## sin limite: usa todos los datos
        ## --- Loss ordinal hibrida (cambio central de exp08) ---
        use_focal_loss  = False,     ## la reemplaza la loss ordinal
        use_ordinal_loss = True,     ## SORD + lambda*QWK
        ordinal_lambda_qwk = 0.3,    ## peso pequeno del componente QWK
        ordinal_distance_power = 1.0,## penalizacion lineal de distancia en SORD
        ordinal_undergrade_beta = 2.0,## exp09: penaliza el sub-diagnostico (beta>1)
        weight_power    = 0.5,       ## class weights suaves (raiz del inverso de frecuencia)
        use_oversampling = False,
        birads_weight   = 1.0,
        density_weight  = 0.5,
        use_official_split = True,    ## usa el split oficial de VinDr (training/test)
    )

print_section('CONFIGURACION DE ENTRENAMIENTO (exp09)', width=70)
print(f'  Dataset: VinDr-Mammo (una sola etapa)')
print(f'  Epocas: {train_config.epochs}, LR: {train_config.lr}')
print(f'  Batch size: {train_config.batch_size}')
print(f'  Resolucion: {train_config.image_height}x{train_config.image_width}')
print(f'  Encoder: {UNFREEZE_LAST_N_BLOCKS} bloques descongelados')
print(f'  Loss ordinal: {train_config.use_ordinal_loss} '
      f'(SORD + {train_config.ordinal_lambda_qwk}*QWK, distance_power={train_config.ordinal_distance_power})')
print(f'  Class weights power: {train_config.weight_power} (suaves)')
print(f'  Oversampling: {train_config.use_oversampling}')
print(f'  Split oficial VinDr: {train_config.use_official_split}')
print(f'  Max muestras: {train_config.max_samples}')
print(f'  Checkpoints: {train_config.checkpoint_dir}')

<h3>5.3 Lanzamiento del entrenamiento</h3>
<p>El entrenamiento se ejecuta como un proceso independiente (nohup) que sobrevive al cierre de la sesion remota. El proceso escribe su progreso en archivos de estado que las celdas de monitoreo leen sin interferir. Esto permite cerrar la sesion SSH, volver despues y retomar el notebook verificando que el entrenamiento haya terminado.</p>

In [ ]:
## Generar la configuracion del runner y lanzar el entrenamiento en background
import json
import subprocess
import os

## Directorio de salida del entrenamiento
TRAIN_OUTPUT_DIR = OUTPUTS_DIR / 'training_run'
TRAIN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Construir la configuracion del runner a partir de train_config (celda 5.2)
runner_config = {
    'output_dir': str(TRAIN_OUTPUT_DIR),
    'checkpoint_path': MAMMOCLIP_CHECKPOINT,
    'efficientnet_name': 'efficientnet-b5',
    'dataset_root': str(DATA_DIR / 'vindr-mammo'),
    'training_config': {
        'epochs':          train_config.epochs,
        'lr':              train_config.lr,
        'batch_size':      train_config.batch_size,
        'num_workers':     train_config.num_workers,
        'weight_decay':    train_config.weight_decay,
        'val_fraction':    train_config.val_fraction,
        'random_seed':     train_config.random_seed,
        'checkpoint_dir':  train_config.checkpoint_dir,
        'device':          'auto',
        'image_height':    train_config.image_height,
        'image_width':     train_config.image_width,
        'max_samples':     train_config.max_samples,
        'resume':          True,   ## reanudar desde checkpoint si se interrumpe
        'unfreeze_last_n_blocks': train_config.unfreeze_last_n_blocks,
        'use_focal_loss':  train_config.use_focal_loss,
        'focal_gamma':     train_config.focal_gamma,
        'weight_power':    train_config.weight_power,
        'use_ordinal_loss': train_config.use_ordinal_loss,
        'ordinal_lambda_qwk': train_config.ordinal_lambda_qwk,
        'ordinal_distance_power': train_config.ordinal_distance_power,
        'ordinal_undergrade_beta': train_config.ordinal_undergrade_beta,
        'use_oversampling': train_config.use_oversampling,
        'birads_weight':   train_config.birads_weight,
        'density_weight':  train_config.density_weight,
        'test_set_dir':    train_config.test_set_dir,
        'use_official_split': train_config.use_official_split,
        'num_birads_classes': 5,
        'num_density_classes': 4,
    },
}

## Escribir la config donde el runner la espera (raiz del proyecto)
config_path = PROJECT_ROOT / 'train_runner_config.json'
with open(config_path, 'w', encoding='utf-8') as f:
    json.dump(runner_config, f, ensure_ascii=False, indent=2)

## Verificar si ya hay un entrenamiento corriendo
status_file = TRAIN_OUTPUT_DIR / 'training_status.json'
pid_file = TRAIN_OUTPUT_DIR / 'training.pid'

ya_corriendo = False
if pid_file.exists():
    try:
        old_pid = int(pid_file.read_text().strip())
        ## Verificar si el proceso sigue vivo
        os.kill(old_pid, 0)
        ya_corriendo = True
        print(f'Ya hay un entrenamiento corriendo (PID {old_pid})')
        print('Usa las celdas de monitoreo para seguir su progreso.')
    except (ProcessLookupError, ValueError):
        ya_corriendo = False

if not ya_corriendo:
    ## Lanzar el runner en background con nohup
    runner_script = PROJECT_ROOT / 'train_runner.py'
    log_file = TRAIN_OUTPUT_DIR / 'nohup.out'

    proc = subprocess.Popen(
        ['nohup', 'python3', str(runner_script)],
        stdout=open(log_file, 'w'),
        stderr=subprocess.STDOUT,
        cwd=str(PROJECT_ROOT),
        start_new_session=True,   ## desacopla el proceso de la sesion SSH
    )
    pid_file.write_text(str(proc.pid))

    print_section('ENTRENAMIENTO LANZADO EN BACKGROUND', width=70)
    print(f'  PID del proceso: {proc.pid}')
    print(f'  Directorio de salida: {TRAIN_OUTPUT_DIR}')
    print(f'  Log: {TRAIN_OUTPUT_DIR / "training.log"}')
    print()
    print('  El entrenamiento corre de forma independiente.')
    print('  Puedes cerrar la sesion remota y volver despues.')
    print('  Usa la celda 5.4 para monitorear el progreso.')

<h3>5.4 Monitoreo del entrenamiento</h3>
<p>Ejecuta esta celda cuantas veces quieras para ver el estado del entrenamiento en background. No interfiere con el proceso. Puedes cerrar la sesion y volver a ejecutarla mas tarde para verificar si termino.</p>

In [ ]:
## Monitorear el estado del entrenamiento en background
import json
import time

## Solo definir la ruta para monitorear (sin lanzar entrenamiento)
TRAIN_OUTPUT_DIR = OUTPUTS_DIR / 'training_run'
print(f'Directorio de entrenamiento: {TRAIN_OUTPUT_DIR}')
print(f'Existe: {TRAIN_OUTPUT_DIR.exists()}')

status_file = TRAIN_OUTPUT_DIR / 'training_status.json'

if not status_file.exists():
    print('Aun no hay estado disponible. El entrenamiento puede estar iniciando.')
    print('Espera unos segundos y vuelve a ejecutar esta celda.')
else:
    with open(status_file, 'r', encoding='utf-8') as f:
        status = json.load(f)

    print_section('ESTADO DEL ENTRENAMIENTO', width=70)
    print(f'  Estado:      {status.get("state", "desconocido")}')
    print(f'  Mensaje:     {status.get("message", "")}')
    print(f'  Actualizado: {status.get("updated_at", "")}')
    print()

    ## Mostrar historial de epocas completadas
    history = status.get('history', {})
    for stage_key in ['stage1', 'stage2']:
        if stage_key in history and history[stage_key]:
            print(f'  {stage_key.upper()}:')
            for ep in history[stage_key]:
                print(f'    Epoca {ep["epoch"]}: '
                      f'train_loss={ep["loss"]:.4f}  '
                      f'val_loss={ep["val_loss"]:.4f}  '
                      f'val_birads_acc={ep["val_birads_acc"]:.4f}')
            print()

    ## Mostrar las ultimas lineas del log
    log_file = TRAIN_OUTPUT_DIR / 'training.log'
    if log_file.exists():
        print('  Ultimas lineas del log:')
        lines = log_file.read_text(encoding='utf-8').splitlines()
        for line in lines[-8:]:
            print(f'    {line}')

    ## Interpretacion del estado
    print()
    estado = status.get('state')
    if estado == 'completed':
        print('  >>> ENTRENAMIENTO COMPLETADO. Ejecuta la celda 5.5 para cargar resultados.')
    elif estado == 'error':
        print('  >>> ERROR EN EL ENTRENAMIENTO. Revisa el traceback:')
        print(status.get('traceback', ''))
    elif estado == 'training':
        print('  >>> Entrenamiento en progreso. Vuelve a ejecutar esta celda para actualizar.')

<h3>5.5 Carga de resultados</h3>
<p>Cuando el entrenamiento termina (estado "completed"), esta celda carga el modelo entrenado y el historial para continuar con la evaluacion. Si cerraste la sesion, al reabrir el notebook ejecuta primero las celdas de inicializacion (1 a 4.1) y luego esta.</p>

In [ ]:
## Cargar el modelo entrenado y el historial cuando el entrenamiento termina
## Definir las rutas necesarias (sin lanzar entrenamiento), para que esta
## celda sea autosuficiente al reabrir la sesion
import json
import torch

TRAIN_OUTPUT_DIR = OUTPUTS_DIR / 'training_run'

status_file = TRAIN_OUTPUT_DIR / 'training_status.json'
final_model_path = TRAIN_OUTPUT_DIR / 'mammovlm_final.pt'
history_path = TRAIN_OUTPUT_DIR / 'training_history.json'

## Variable global para la duracion (se usa al registrar el experimento)
training_duration_s = None
training_duration_str = ''

if not status_file.exists():
    print('No hay entrenamiento registrado. Ejecuta la celda 5.3 primero.')
else:
    with open(status_file, 'r', encoding='utf-8') as f:
        status = json.load(f)

    if status.get('state') != 'completed':
        print(f'El entrenamiento aun no termina (estado: {status.get("state")}).')
        print('Usa la celda 5.4 para monitorear hasta que el estado sea "completed".')
    else:
        ## Cargar pesos entrenados en el modelo
        checkpoint = torch.load(final_model_path, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        model.eval()

        ## Cargar historial (nuevo formato: incluye duracion)
        with open(history_path, 'r', encoding='utf-8') as f:
            history_data = json.load(f)

        ## Compatibilidad: el historial puede venir como dict directo (formato viejo)
        ## o como {"history": ..., "training_duration_s": ...} (formato nuevo)
        if 'history' in history_data and 'training_duration_s' in history_data:
            history = history_data['history']
            training_duration_s = history_data.get('training_duration_s')
            training_duration_str = history_data.get('training_duration_str', '')
        else:
            history = history_data

        print_section('MODELO ENTRENADO CARGADO', width=70)
        print(f'  Modelo: {final_model_path.name}')
        if training_duration_str:
            print(f'  Duracion del entrenamiento: {training_duration_str}')
        print(f'  El modelo esta listo para evaluacion (Areas 1-4 con datos reales)')
        print()

        ## Resumen del entrenamiento
        for stage_key in ['stage1', 'stage2']:
            if stage_key in history and history[stage_key]:
                ultima = history[stage_key][-1]
                print(f'  {stage_key.upper()} (epoca final {ultima["epoch"]}): '
                      f'val_birads_acc={ultima["val_birads_acc"]:.4f}')

<h3>5.6 Curvas de aprendizaje</h3>

In [ ]:
## Visualizar las curvas de entrenamiento por etapa
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for stage_key, ax in zip(['stage1', 'stage2'], axes):
    if stage_key not in history or not history[stage_key]:
        ax.text(0.5, 0.5, f'{stage_key} sin datos', ha='center', va='center')
        ax.axis('off')
        continue

    h = history[stage_key]
    epochs = [e['epoch'] for e in h]
    train_loss = [e['loss'] for e in h]
    val_loss = [e['val_loss'] for e in h]
    val_acc = [e['val_birads_acc'] for e in h]

    ax.plot(epochs, train_loss, 'o-', label='Train Loss', color='steelblue')
    ax.plot(epochs, val_loss, 's-', label='Val Loss', color='coral')
    ax.set_xlabel('Epoca')
    ax.set_ylabel('Loss')
    ax.set_title(f'{stage_key.upper()}: Curvas de Loss')
    ax.legend(loc='upper left')
    ax.grid(True, alpha=0.3)

    ax2 = ax.twinx()
    ax2.plot(epochs, val_acc, '^--', label='Val BI-RADS Acc', color='green')
    ax2.set_ylabel('Accuracy')
    ax2.set_ylim(0, 1)
    ax2.legend(loc='upper right')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Curvas guardadas: training_curves.png')

<h3>5.7 Evaluación en test set</h3>
<p>El test set se separo antes del entrenamiento (15% estratificado por dataset y BI-RADS) y nunca se uso para entrenar ni validar. Esta evaluacion sobre datos no vistos es la metrica real y defendible del modelo. Se aplican las metricas clinicas de Area 3 sobre las predicciones del modelo entrenado.</p>

In [ ]:
## Evaluar el modelo entrenado sobre el test set reservado de VinDr
## Esta es la evaluacion metodologicamente valida para la tesis
import pandas as pd
import numpy as np
import torch
from data_loading import MammoDataset, MammoCLIPTransform, index_to_birads
from torch.utils.data import DataLoader

## Cargar el test set reservado (split oficial de VinDr, guardado en entrenamiento)
test_dir = Path(train_config.test_set_dir)
test_path = test_dir / 'test_set_vindr.csv'

if not test_path.exists():
    print('No se encontro el test set de VinDr.')
    print('Asegurate de haber corrido el entrenamiento completo (PRUEBA_DE_HUMO=False).')
else:
    test_records = pd.read_csv(test_path)
    print(f'  Test set VinDr: {len(test_records)} muestras')
    print()

    ## Transform de evaluacion (alta resolucion, sin augmentation)
    eval_transform = MammoCLIPTransform(
        height=IMAGE_HEIGHT, width=IMAGE_WIDTH, augment=False, use_clahe=True
    )

    ## Dataset y loader del test
    test_ds = MammoDataset(test_records, eval_transform, augment=False)
    test_loader = DataLoader(test_ds, batch_size=8, shuffle=False, num_workers=4)

    ## Recolectar predicciones del modelo sobre el test set
    model.eval()
    y_true, y_pred, y_probs = [], [], []

    print('  Evaluando modelo sobre el test set...')
    with torch.no_grad():
        for batch in test_loader:
            images = batch['image'].to(device)
            outputs = model(images)
            probs = torch.softmax(outputs['birads'], dim=1)
            preds = torch.argmax(probs, dim=1)

            y_true.extend(batch['birads'].numpy().tolist())
            y_pred.extend(preds.cpu().numpy().tolist())
            y_probs.extend(probs.cpu().numpy().tolist())

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_probs = np.array(y_probs)

    print(f'  Predicciones recolectadas: {len(y_true)}')
    print()

    ## Aplicar las metricas clinicas de Area 3 sobre las predicciones reales
    ## NOTA: en exp06 BI-RADS tiene 5 clases (1-5, indices 0-4), no 6 como antes.
    ## Los indices 0-4 corresponden a BI-RADS 1-5.
    medical_report_gen = MedicalMetricsReport(num_classes=5)
    test_report = medical_report_gen.compute_full_report(y_true, y_pred, y_probs)
    print(medical_report_gen.generate_summary(test_report))

    ## Guardar el reporte de evaluacion en test
    test_eval_path = RESULTS_DIR / f'test_set_evaluation_{get_timestamp()}.json'
    save_json(test_report, test_eval_path)
    print(f'Reporte de evaluacion en test guardado: {test_eval_path.name}')

<h3>5.8 Threshold tuning clínico</h3>
<p>Ajuste del umbral de decision benigno/maligno para recuperar sensibilidad clinica. El umbral optimo (indice de Youden) se elige sobre el conjunto de <b>validacion</b> y se aplica fijo al de <b>test</b>, que es la forma metodologicamente correcta (elegirlo sobre el test inflaria el resultado). Produce dos reportes para la tesis: clasificacion multiclase por argmax (celda 5.7) y cribado de alta sensibilidad por umbral (esta celda).</p>

In [ ]:
## CAMINO B: threshold tuning con umbral elegido en VALIDACION, aplicado a TEST
## Reutiliza el modelo, el transform, y las predicciones de test de la celda 5.7
## (model, eval_transform, device, y_true, y_probs, train_config deben existir)
from train import separate_test_set_vindr, split_train_val
from data_loading import load_vindr_records
from threshold_tuning import find_optimal_threshold, apply_threshold, compare_argmax_vs_threshold

print_section('5.8 THRESHOLD TUNING CLÍNICO', width=70)

## --- 1. Reconstruir el MISMO split de validacion que vio el entrenamiento ---
## Cargar VinDr, separar el test (split oficial), aplicar split_train_val con la
## misma semilla y val_fraction que uso train_config.
VINDR_ROOT = str(DATA_DIR / 'vindr-mammo')
all_records = load_vindr_records(VINDR_ROOT)
train_val_df, _ = separate_test_set_vindr(all_records, train_config, test_path)
_, val_df = split_train_val(train_val_df, train_config.val_fraction, train_config.random_seed)
print(f'  Split de validacion reconstruido: {len(val_df)} muestras')

## --- 2. Predecir sobre validacion (reusa model y eval_transform de la 5.7) ---
val_ds = MammoDataset(val_df, eval_transform, augment=False)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=4)

model.eval()
val_true, val_probs = [], []
print('  Evaluando modelo sobre validacion...')
with torch.no_grad():
    for batch in val_loader:
        images = batch['image'].to(device)
        outputs = model(images)
        probs = torch.softmax(outputs['birads'], dim=1)
        val_true.extend(batch['birads'].numpy().tolist())
        val_probs.extend(probs.cpu().numpy().tolist())
val_true = np.array(val_true)
val_probs = np.array(val_probs)

## --- 3. Elegir el umbral optimo (Youden J) en VALIDACION ---
opt_threshold, val_best, _ = find_optimal_threshold(val_probs, val_true)
print(f'  Umbral optimo (elegido en validacion): {opt_threshold:.4f}')
print()

## --- 4. Aplicar ese umbral fijo al TEST (resultado honesto) ---
comp = compare_argmax_vs_threshold(y_probs, y_true, opt_threshold)

print('DETECCION BINARIA benigno (BR1-3) vs maligno (BR4-5) en TEST:')
print()
print('  Metodo ARGMAX (reporte multiclase, celda 9.7):')
print(f'    Sensibilidad:  {comp["argmax"]["sensitivity"]:.4f}')
print(f'    Especificidad: {comp["argmax"]["specificity"]:.4f}')
print(f'    Falsos negativos: {comp["argmax"]["fn"]}')
print()
print(f'  Metodo UMBRAL clinico ({opt_threshold:.3f}, elegido en validacion):')
print(f'    Sensibilidad:  {comp["threshold"]["sensitivity"]:.4f}')
print(f'    Especificidad: {comp["threshold"]["specificity"]:.4f}')
print(f'    Falsos negativos: {comp["threshold"]["fn"]}')
print(f'    Falsos positivos: {comp["threshold"]["fp"]}')
print(f'    VPP: {comp["threshold"]["ppv"]:.4f}')
print()
print(f'  Cambio en sensibilidad: {comp["argmax"]["sensitivity"]:.4f} -> '
      f'{comp["threshold"]["sensitivity"]:.4f} '
      f'({comp["threshold"]["sensitivity"] - comp["argmax"]["sensitivity"]:+.4f})')

## --- 5. Guardar el bloque de threshold tuning junto al reporte ---
threshold_report = {
    'optimal_threshold': opt_threshold,
    'chosen_on': 'validation',
    'n_validation': int(len(val_df)),
    'validation_metrics_at_threshold': val_best,
    'test_argmax_vs_threshold': comp,
}
threshold_path = RESULTS_DIR / f'threshold_tuning_camino_b_{get_timestamp()}.json'
save_json(threshold_report, threshold_path)
print()
print(f'Reporte de threshold tuning guardado: {threshold_path.name}')

<hr/>
<h2 id='experimentos'>6. Historial de Experimentos</h2>
<p>Sistema de registro de experimentos para documentar la evolucion del modelo a lo largo de la tesis. Cada experimento se guarda con su configuracion, metricas sobre el test set, hipotesis y resultados. Esto permite presentar en la defensa un analisis comparativo de todas las iteraciones realizadas.</p>

<h3>6.1 Registro del experimento actual</h3>

In [ ]:
## Registrar el experimento actual en el historial
## El tracker guarda cada experimento sin sobreescribir los anteriores
tracker = ExperimentTracker(OUTPUTS_DIR / 'experiments')

## --- CAMPOS MANUALES (completar para cada experimento) ---
## Identificador unico del experimento (coincide con EXPERIMENT_NAME de la celda 4.1)
EXPERIMENT_ID = EXPERIMENT_NAME

## Hipotesis: que se probo y por que (esto es clave para la tesis)
HYPOTHESIS = (
    'Sobre la base de exp08 (el mejor modelo: Mammo-CLIP + loss ordinal SORD+QWK '
    '+ descongelamiento 2 bloques), se introduce UN solo cambio: SORD ASIMETRICO. '
    'La matriz de distancia de SORD penaliza el sub-diagnostico (predecir por '
    'debajo de la clase real, el falso negativo clinico) con un factor beta=2.0, '
    'mientras el sobre-diagnostico mantiene penalizacion normal. Todo lo demas se '
    'mantiene identico a exp08 (lambda=0.3, weight_power=0.5, 2 bloques) para '
    'aislar el efecto de la asimetria. Hipotesis: la asimetria sesga las soft '
    'labels de los casos BR4-5 hacia arriba, subiendo la sensibilidad de malignos '
    'DIRECTAMENTE en el entrenamiento (sin depender del threshold tuning), '
    'idealmente con menos falsos positivos que el ajuste de umbral de exp08.'
)

## Notas adicionales (observaciones, limitaciones detectadas)
NOTES = (
    'exp09: SORD asimetrico (beta=2.0) como unico cambio sobre exp08. '
    '[COMPLETAR tras evaluar: comparar contra exp08. Metricas clave a vigilar: '
    'sensibilidad de malignos (objetivo: subir desde 0.37 del argmax de exp08 '
    'SIN depender del threshold tuning), Quadratic Kappa (no debe colapsar desde '
    '0.48), Cohen Kappa (desde 0.26), y falsos positivos (idealmente menos que '
    'los 1138 del threshold tuning de exp08). Si no mejora a exp08, exp08 + '
    'threshold validado queda como modelo definitivo.]'
)
## --- FIN CAMPOS MANUALES ---

## Configuracion tecnica (automatica)
config_dict = {
    'encoder': 'Mammo-CLIP (EfficientNet-B5)',
    'dataset': 'VinDr-Mammo',
    'freeze_encoder': UNFREEZE_LAST_N_BLOCKS == 0,
    'unfreeze_last_n_blocks': train_config.unfreeze_last_n_blocks,
    'epochs': train_config.epochs,
    'lr': train_config.lr,
    'batch_size': train_config.batch_size,
    'image_resolution': f'{train_config.image_height}x{train_config.image_width}',
    'num_birads_classes': 5,
    'num_density_classes': 4,
    'tiling': False,
    'single_stage': True,
    'use_focal_loss': train_config.use_focal_loss,
    'use_ordinal_loss': train_config.use_ordinal_loss,
    'ordinal_lambda_qwk': train_config.ordinal_lambda_qwk,
    'ordinal_distance_power': train_config.ordinal_distance_power,
    'ordinal_undergrade_beta': train_config.ordinal_undergrade_beta,
    'weight_power': train_config.weight_power,
    'use_oversampling': train_config.use_oversampling,
    'use_official_split': train_config.use_official_split,
}

## Registrar (test_report y history provienen de las celdas 5.5 y 5.7)
exp_dir = tracker.register_experiment(
    experiment_id=EXPERIMENT_ID,
    hypothesis=HYPOTHESIS,
    config_dict=config_dict,
    test_metrics=test_report,
    training_history=history,
    model_path=str(TRAIN_OUTPUT_DIR / 'mammovlm_final.pt'),
    notes=NOTES,
)

print_section('EXPERIMENTO REGISTRADO', width=70)
print(f'  ID: {EXPERIMENT_ID}')
print(f'  Guardado en: {exp_dir}')
print(f'  Incluye: configuracion, metricas, historial, modelo, hipotesis')

<h3>6.2 Comparativa de experimentos</h3>

In [ ]:
## Mostrar el historial completo de experimentos
## Esta tabla es la que se presenta en la tesis para mostrar la evolucion
tracker.print_comparison()

<h3>6.3 Evolución de métricas</h3>

In [ ]:
## Visualizar la evolucion de las metricas clave a traves de los experimentos
## (util cuando hay 2 o mas experimentos para comparar)
comparison = tracker.get_comparison_table()

if len(comparison) < 2:
    print('Se necesitan al menos 2 experimentos para graficar la evolucion.')
    print(f'Actualmente hay {len(comparison)} experimento(s) registrado(s).')
    print('Despues de los proximos entrenamientos, esta celda mostrara la comparacion.')
else:
    exp_ids = [e['experiment_id'] for e in comparison]
    metricas_clave = ['accuracy', 'macro_f1', 'quadratic_kappa', 'auc_macro',
                      'sensitivity', 'cohen_kappa']

    fig, ax = plt.subplots(figsize=(13, 6))
    x = np.arange(len(exp_ids))
    width = 0.13

    for i, metrica in enumerate(metricas_clave):
        valores = [e.get(metrica, 0) for e in comparison]
        ax.bar(x + i * width, valores, width, label=metrica)

    ax.set_xticks(x + width * (len(metricas_clave) - 1) / 2)
    ax.set_xticklabels([eid.replace('_', '\n') for eid in exp_ids], fontsize=8)
    ax.set_ylabel('Valor de la metrica')
    ax.set_title('Evolucion de Metricas entre Experimentos')
    ax.legend(loc='upper left', fontsize=8, ncol=2)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 1)

    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'experiments_evolution.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Grafica guardada: experiments_evolution.png')

In [ ]:
print_section('RESUMEN EJECUTIVO - MammoVLM V2', width=70)
print()
print(f'Ejecutado: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print()

print('PIPELINE RAG:')
if indexer is not None:
    stats = indexer.get_index_stats()
    print(f'  Estado: {stats["status"]}')
    print(f'  Chunks indexados: {stats["total_chunks"]}')
    print(f'  Fuentes: {len(stats.get("chunks_per_source", {}))}')
    print(f'  Modelo embeddings: NeuML/pubmedbert-base-embeddings')
else:
    print('  Estado: No disponible (literatura no encontrada)')
print()

print('VOCABULARIO MEDICO:')
print(f'  Descriptores BI-RADS: {len(birads_lexicon.get_all_descriptors())}')
print()

print('ESTADO DE COMPONENTES:')
print('  Pipeline RAG (Literatura medica):       COMPLETADO')
print('  Modelo (exp08, Mammo-CLIP + VinDr):     COMPLETADO')
print('  Entrenamiento y evaluacion (sec 9):     COMPLETADO')
print('  Pipeline VLM end-to-end (sec 11):       COMPLETADO')
print('  Evaluacion informes sin referencia:     COMPLETADO')
print()
print_separator('=', 70)


<hr/>
<h2 id='pipeline'>7. Inferencia Diagnóstica End-to-End</h2>
<p>Esta seccion ejecuta el flujo completo del modelo de lenguaje-vision en el orden de inferencia real:</p>
<p style="text-align:center"><b>imagen &rarr; encoder (Mammo-CLIP) &rarr; clasificacion (BI-RADS + densidad) &rarr; decoder (LLM + RAG) &rarr; informe diagnostico</b></p>
<p>El informe se genera a partir de la prediccion por <code>argmax</code> (BI-RADS multiclase) e incorpora el score de malignidad P(BR4)+P(BR5). El decoder (Qwen2.5) redacta el informe estructurado segun ACR BI-RADS, fundamentado en la literatura medica recuperada por RAG, sin fabricar hallazgos que el modelo no predice.</p>

<h3>7.1 Setup autónomo</h3>
<p>Define las variables base (rutas, constantes del modelo, utilidades y test set) necesarias para la sección 7, sin depender de haber ejecutado las secciones 1-6. Si esas variables ya existen en memoria (porque se corrio el notebook completo), se respetan; si no, se definen aqui con los valores del proyecto.</p>

In [ ]:
## Setup minimo autonomo para la seccion 11 (pipeline VLM end-to-end)
## Replica lo imprescindible sin depender de las secciones 1-6. Las variables
## que ya existan en memoria NO se sobreescriben (se usan tal cual).
import sys, re, json
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd

## --- Rutas base ---
if 'PROJECT_ROOT' not in dir():
    PROJECT_ROOT = Path.cwd()
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))
if 'DATA_DIR' not in dir():
    DATA_DIR = PROJECT_ROOT / 'data'
if 'OUTPUTS_DIR' not in dir():
    OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
if 'RESULTS_DIR' not in dir():
    RESULTS_DIR = PROJECT_ROOT / 'results'
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
if 'LITERATURE_DIR' not in dir():
    LITERATURE_DIR = PROJECT_ROOT / 'Libros'
if 'RAG_INDEX_DIR' not in dir():
    RAG_INDEX_DIR = DATA_DIR / 'rag_index'
if 'TRAIN_OUTPUT_DIR' not in dir():
    TRAIN_OUTPUT_DIR = OUTPUTS_DIR / 'training_run'

## --- Constantes del modelo (valores de exp08) ---
if 'MAMMOCLIP_CHECKPOINT' not in dir():
    MAMMOCLIP_CHECKPOINT = str(PROJECT_ROOT / 'models' / 'mammo_clip_b5.tar')
if 'UNFREEZE_LAST_N_BLOCKS' not in dir():
    UNFREEZE_LAST_N_BLOCKS = 2
if 'IMAGE_HEIGHT' not in dir():
    IMAGE_HEIGHT = 1520
if 'IMAGE_WIDTH' not in dir():
    IMAGE_WIDTH = 912

## Identificador del experimento definitivo (todo el pipeline se relaciona con el)
if 'EXPERIMENT_FINAL' not in dir():
    EXPERIMENT_FINAL = 'exp08_ordinal_sord_qwk_descongelado'

## --- Utilidades (fallback si no se importaron desde utils) ---
if 'get_timestamp' not in dir():
    def get_timestamp():
        return datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
if 'print_section' not in dir():
    def print_section(titulo, width=70):
        print('=' * width)
        print(titulo.center(width))
        print('=' * width)
if 'save_json' not in dir():
    def save_json(data, path):
        with open(path, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=2)

## --- Localizar el test set (lo lee de disco si no esta en memoria) ---
if 'test_path' not in dir():
    _candidatos = [
        TRAIN_OUTPUT_DIR.parent / 'test_sets' / 'test_set_vindr.csv',
        OUTPUTS_DIR / 'test_sets' / 'test_set_vindr.csv',
        DATA_DIR / 'test_sets' / 'test_set_vindr.csv',
    ]
    test_path = next((str(p) for p in _candidatos if p.exists()), None)
    if test_path is None:
        raise FileNotFoundError(
            'No se encontro test_set_vindr.csv. Verifique la ruta o ejecute la seccion 9.'
        )

print_section('SETUP AUTONOMO SECCION 11', width=70)
print(f'  PROJECT_ROOT: {PROJECT_ROOT}')
print(f'  Checkpoint Mammo-CLIP: {MAMMOCLIP_CHECKPOINT}')
print(f'  Modelo entrenado: {TRAIN_OUTPUT_DIR / "mammovlm_final.pt"}')
print(f'  Resolucion: {IMAGE_HEIGHT}x{IMAGE_WIDTH}, bloques descongelados: {UNFREEZE_LAST_N_BLOCKS}')
print(f'  Literatura RAG: {LITERATURE_DIR}')
print(f'  Test set: {test_path}')
print('  Listo para ejecutar 7.2 (cargar modelo) en adelante.')

<h3>7.2 Cargar el clasificador</h3>
<p>Carga el mejor modelo del proyecto (exp08) con su encoder Mammo-CLIP y las cabezas de clasificacion BI-RADS + densidad.</p>

In [ ]:
## Cargar el modelo MammoVLM entrenado (encoder + clasificacion)
## MODELO DEFINITIVO: exp08 (mejor clasificador, base del modelo final con threshold)
import torch
from models import MammoVLM

## EXPERIMENT_FINAL viene del setup 7.1 (exp08). Localizar su checkpoint. Prioridad:
##   1. La copia que el ExperimentTracker guardo al registrar exp08
##      (OUTPUTS_DIR/experiments/<exp08>/model.pt) -> es el modelo correcto de exp08
##   2. El mammovlm_final.pt generico (solo si coincide con exp08; ojo si se
##      corrieron experimentos posteriores que lo sobreescribieron)
_exp_model = OUTPUTS_DIR / 'experiments' / EXPERIMENT_FINAL / 'model.pt'
_generic_model = TRAIN_OUTPUT_DIR / 'mammovlm_final.pt'

## Verificacion explicita: avisar de donde viene el modelo
if not _exp_model.exists():
    print('AVISO: no se encontro el modelo en la carpeta del experimento:')
    print(f'  {_exp_model}')
    print('  Esto ocurre si exp08 no fue registrado con la celda 10.1, o si la')
    print('  ruta de experiments difiere. Se intentara usar mammovlm_final.pt,')
    print('  PERO verifique que ese archivo corresponda a exp08 y no a exp09.')
    print()

if _exp_model.exists():
    VLM_CHECKPOINT = str(_exp_model)
    _fuente = f'carpeta del experimento {EXPERIMENT_FINAL}'
else:
    VLM_CHECKPOINT = str(_generic_model)
    _fuente = 'mammovlm_final.pt (ADVERTENCIA: verifique que sea exp08, no un experimento posterior)'

device = 'cuda' if torch.cuda.is_available() else 'cpu'

## Construir la arquitectura con la configuracion de exp08 (2 bloques descongelados)
vlm_model = MammoVLM(
    checkpoint_path=MAMMOCLIP_CHECKPOINT,
    efficientnet_name='efficientnet-b5',
    num_birads_classes=5,
    num_density_classes=4,
    freeze_encoder=True,
    unfreeze_last_n_blocks=UNFREEZE_LAST_N_BLOCKS,
)
_ckpt = torch.load(VLM_CHECKPOINT, map_location=device, weights_only=False)
vlm_model.load_state_dict(_ckpt['model_state_dict'])
vlm_model = vlm_model.to(device)
vlm_model.eval()

print_section('MODELO VLM CARGADO (encoder + clasificacion)', width=70)
print(f'  Experimento: {EXPERIMENT_FINAL}')
print(f'  Checkpoint: {VLM_CHECKPOINT}')
print(f'  Fuente: {_fuente}')
print(f'  Dispositivo: {device}')
print(f'  Parametros totales: {vlm_model.get_total_parameters():,}')

<h3>7.3 Cargar el generador de informes</h3>
<p>Carga el modelo de lenguaje que redacta el informe y el retriever de literatura medica (indice RAG ya construido y guardado en disco). La carga del LLM de 7B requiere VRAM; puede tardar algunos minutos.</p>

In [ ]:
## Cargar el LLM (decoder) y el retriever RAG ya construido
from report_generator import load_llm_for_generation, ReportGenerator
from rag import create_rag_pipeline

## --- Cargar el retriever RAG desde el indice guardado (force_rebuild=False) ---
## Usa LITERATURE_DIR y RAG_INDEX_DIR ya definidos en la celda de configuracion
print('Cargando indice RAG (desde cache)...')
rag_indexer, rag_retriever = create_rag_pipeline(
    pdf_dir=str(LITERATURE_DIR),
    index_dir=str(RAG_INDEX_DIR),
    device='auto',
    force_rebuild=False,   ## carga el indice guardado, no lo reconstruye
)
print(f'  RAG listo: {rag_indexer.get_index_stats().get("total_chunks", 0)} chunks')

## --- Cargar el LLM Qwen2.5 (decoder de informes) ---
print('\nCargando LLM Qwen2.5-7B (puede tardar)...')
llm_model, llm_tokenizer = load_llm_for_generation(
    model_name='Qwen/Qwen2.5-7B-Instruct',
    device='auto',
    dtype='bfloat16',
)

## --- Construir el generador de informes (decoder + RAG) ---
report_generator = ReportGenerator(
    retriever=rag_retriever,
    llm=llm_model,
    tokenizer=llm_tokenizer,
    language='es',
    use_rag=True,
    rag_top_k=3,
)
print_section('DECODER LISTO (LLM + RAG)', width=70)
print('  Generador de informes configurado con LLM Qwen2.5 + RAG')

<h3>7.4 Inferencia sobre una imagen</h3>
<p>Flujo completo sobre una sola imagen: se carga, pasa por el encoder y la clasificacion (argmax para BI-RADS y densidad), se calcula el score de malignidad P(BR4)+P(BR5), y el decoder genera el informe.</p>

In [ ]:
## Funcion de inferencia end-to-end: imagen -> ... -> informe
import numpy as np
import matplotlib.pyplot as plt
from data_loading import MammoCLIPTransform, load_image_as_pil

## Transform de inferencia (alta resolucion, sin augmentation)
infer_transform = MammoCLIPTransform(
    height=IMAGE_HEIGHT, width=IMAGE_WIDTH, augment=False, use_clahe=True
)

def generar_informe_para_imagen(image_path, mostrar=True):
    ##
    ## Pipeline completo: imagen -> encoder -> clasificacion -> decoder -> informe
    ##
    ## 1. Cargar y preprocesar la imagen
    pil_img = load_image_as_pil(image_path)
    img_tensor = infer_transform(pil_img).unsqueeze(0).to(device)

    ## 2. Encoder + clasificacion (argmax)
    vlm_model.eval()
    with torch.no_grad():
        outputs = vlm_model(img_tensor)
        birads_probs = torch.softmax(outputs['birads'][0], dim=-1)
        density_probs = torch.softmax(outputs['density'][0], dim=-1)

    birads_idx = int(torch.argmax(birads_probs).item())   ## 0-4 (argmax)
    density_idx = int(torch.argmax(density_probs).item())  ## 0-3
    birads_conf = float(birads_probs[birads_idx].item())

    ## 3. Score de malignidad = P(BR4) + P(BR5) = prob de indices 3 y 4
    malignancy_score = float(birads_probs[3].item() + birads_probs[4].item())

    ## 4. Construir la prediccion para el decoder
    prediction = {
        'birads_pred': birads_idx,          ## indice 0-4 (el generador lo pasa a 1-5)
        'birads_confidence': birads_conf,
        'density_pred': density_idx,
        'malignancy_score': malignancy_score,
    }

    ## 5. Decoder: generar el informe (LLM + RAG)
    result = report_generator.generate(prediction, max_new_tokens=500)

    if mostrar:
        ## --- Mostrar la imagen mamografica (escala de grises) ---
        fig, ax = plt.subplots(figsize=(5, 7))
        ax.imshow(np.array(pil_img), cmap='gray')
        ax.set_title(f'Mamografia - BI-RADS {birads_idx+1} (confianza {birads_conf:.2f})',
                     fontsize=11)
        ax.axis('off')
        plt.tight_layout()
        plt.show()

        ## --- Mostrar el informe debajo de la imagen ---
        print_section(f'INFORME GENERADO - BI-RADS {birads_idx+1}', width=70)
        print(f'Imagen: {image_path}')
        print(f'Clasificacion: BI-RADS {birads_idx+1} (confianza {birads_conf:.2f}), '
              f'densidad indice {density_idx}')
        print(f'Score malignidad P(BR4)+P(BR5): {malignancy_score:.3f}')
        print(f'Fragmentos RAG usados: {result["rag_chunks_used"]} '
              f'(fuentes: {result["rag_sources"]})')
        print()
        print('-' * 70)
        print(result['report'])
        print('-' * 70)

    return result

## Probar con una imagen del test set
_ejemplo = pd.read_csv(test_path).iloc[0]
_ = generar_informe_para_imagen(_ejemplo['image_path'])

<h3>7.5 Generación de informes por lotes</h3>
<p>Demostracion del pipeline sobre una muestra de imagenes del test set, cubriendo distintas categorias BI-RADS para ilustrar la variedad de informes.</p>

In [ ]:
## Generar informes para una muestra de imagenes del test (una por categoria BI-RADS)
test_df_full = pd.read_csv(test_path)

## Elegir hasta una imagen por cada categoria BI-RADS (1-5) para variedad
ejemplos = []
for b in [1, 2, 3, 4, 5]:
    subset = test_df_full[test_df_full['birads'] == b]
    if len(subset) > 0:
        ejemplos.append(subset.iloc[0])

print(f'Generando informes para {len(ejemplos)} imagenes (una por categoria BI-RADS)...')
print()

informes_generados = []
for row in ejemplos:
    res = generar_informe_para_imagen(row['image_path'], mostrar=True)
    res['birads_real'] = int(row['birads'])
    informes_generados.append(res)
    print()

## Guardar los informes generados
informes_serializables = [
    {
        'birads_real': r['birads_real'],
        'birads_predicho': r['birads_level'],
        'densidad': r['density_description'],
        'score_malignidad': r['malignancy_score'],
        'recomendacion': r['recommendation'],
        'informe': r['report'],
        'rag_fuentes': r['rag_sources'],
    }
    for r in informes_generados
]
informes_path = RESULTS_DIR / f'informes_generados_{EXPERIMENT_FINAL}_{get_timestamp()}.json'
save_json(informes_serializables, informes_path)
print(f'Informes guardados: {informes_path.name}')

<hr/>
<h2 id='evaluacion'>8. Evaluación del Generador de Informes</h2>
<p>Evalua la calidad de los informes generados por el LLM SIN necesitar informes de referencia (VinDr no los tiene). Mide cuatro dimensiones:</p>
<ul>
<li><b>Fidelidad factual</b>: el BI-RADS escrito en el informe coincide con la prediccion del modelo (lo mas critico para un VLM medico).</li>
<li><b>Coherencia clinica</b>: usando el CoherenceValidator (mencion BI-RADS, coherencia de recomendacion, completitud).</li>
<li><b>Adherencia terminologica</b>: uso de vocabulario ACR BI-RADS valido.</li>
<li><b>Seguridad</b>: informe en espanol, sin fuga de idioma, completo.</li>
</ul>

<h3>8.1 Setup autónomo</h3>
<p>Esta celda prepara lo imprescindible para la evaluacion sin depender de haber ejecutado las secciones 1-6. Solo requiere que el modelo (7.2) y el generador (7.3) esten cargados en memoria, ya que son objetos pesados en GPU que no pueden recrearse automaticamente.</p>

In [ ]:
## Setup minimo autonomo para la evaluacion del LLM
## Replica solo lo imprescindible (imports, rutas, test set) sin depender de
## las secciones 1-6. Verifica que los objetos pesados esten cargados.
import sys, re, json
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd

## Rutas (autonomas: no dependen de la celda 4)
if 'PROJECT_ROOT' not in dir():
    PROJECT_ROOT = Path.cwd()
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))
if 'DATA_DIR' not in dir():
    DATA_DIR = PROJECT_ROOT / 'data'
if 'RESULTS_DIR' not in dir():
    RESULTS_DIR = PROJECT_ROOT / 'results'
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## get_timestamp autonomo (por si no se importo desde utils)
if 'get_timestamp' not in dir():
    def get_timestamp():
        return datetime.now().strftime('%Y-%m-%d_%H-%M-%S')

## print_section autonomo (por si no se importo desde utils)
if 'print_section' not in dir():
    def print_section(titulo, width=70):
        print('=' * width)
        print(titulo.center(width))
        print('=' * width)

## save_json autonomo (por si no se importo desde utils)
if 'save_json' not in dir():
    def save_json(data, path):
        with open(path, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=2)

## Localizar el test set (autonomo: lo lee de disco si no esta en memoria)
if 'test_path' not in dir():
    _candidatos = [
        PROJECT_ROOT / 'outputs' / 'test_sets' / 'test_set_vindr.csv',
        DATA_DIR / 'test_sets' / 'test_set_vindr.csv',
    ]
    test_path = next((str(p) for p in _candidatos if p.exists()), None)
    if test_path is None:
        raise FileNotFoundError(
            'No se encontro test_set_vindr.csv. Ejecute la celda 5.7 o verifique la ruta.'
        )
print(f'Test set: {test_path}')

## Verificar que los objetos pesados esten cargados (no se pueden recrear solos)
_faltantes = []
if 'vlm_model' not in dir():
    _faltantes.append('vlm_model (ejecute la celda 7.2)')
if 'report_generator' not in dir():
    _faltantes.append('report_generator (ejecute la celda 7.3)')
if 'generar_informe_para_imagen' not in dir():
    _faltantes.append('generar_informe_para_imagen (ejecute la celda 7.4)')

if _faltantes:
    print('\n*** FALTAN OBJETOS EN MEMORIA ***')
    for f in _faltantes:
        print(f'  - {f}')
    print('\nEjecute esas celdas antes de continuar con la evaluacion.')
else:
    print('Objetos necesarios en memoria: OK (modelo, generador, funcion de inferencia)')
    print('Listo para evaluar.')

<h3>8.2 Evaluación factual del generador</h3>
<p>Genera informes para una muestra del test set y calcula las metricas de fidelidad, coherencia, terminologia y seguridad de forma agregada.</p>

In [ ]:
## Evaluacion sin referencia del generador de informes
from coherence_metrics import PredictionRecord, CoherenceValidator

## --- Funciones auxiliares de evaluacion ---
def extraer_birads_del_texto(texto):
    ## Extrae la categoria BI-RADS mencionada en el informe (para fidelidad factual)
    ##
    ## Robusto a las formas naturales que usa el LLM, p.ej.:
    ##   "BI-RADS 5", "BI-RADS: 5", "categoria BI-RADS asignada es 5",
    ##   "la categoria BI-RADS es 4", "BI-RADS 4A" (toma el 4)
    ## Estrategia: permitir hasta ~25 caracteres (palabras como "asignada es")
    ## entre el token BI-RADS y el digito. Se prioriza la seccion IMPRESION.
    ##
    def _buscar(t):
        ## Permite palabras intermedias entre BI-RADS y el numero (asignada es, etc.)
        patrones = [
            r"BI[\s\-]?RADS[^0-9]{0,25}?([0-5])",     ## BI-RADS ... N (con texto en medio)
            r"categoria[^0-9]{0,25}?([0-5])",           ## categoria ... N
        ]
        for pat in patrones:
            m = re.search(pat, t, re.IGNORECASE)
            if m:
                return int(m.group(1))
        return None

    ## Priorizar la seccion IMPRESION (donde se asienta la categoria oficial)
    m_imp = re.search(r"IMPRESION(.*?)(?:RECOMENDACION|$)", texto,
                      re.IGNORECASE | re.DOTALL)
    if m_imp:
        r = _buscar(m_imp.group(1))
        if r is not None:
            return r

    ## Si no aparece en IMPRESION, buscar en todo el texto
    return _buscar(texto)

def contiene_no_latino(texto):
    ## Detecta fuga de idioma (caracteres CJK)
    for ch in texto:
        c = ord(ch)
        if 0x4E00 <= c <= 0x9FFF or 0x3040 <= c <= 0x30FF or 0xAC00 <= c <= 0xD7AF:
            return True
    return False

## Morfologias que el modelo NO predice: si aparecen, es alucinacion
_TERMINOS_ALUCINACION = [
    "espiculada", "espiculado", "lobulada", "circunscrita", "irregular",
    "microcalcificaciones", "pleomorfica", "cuadrante", "region retroareolar",
]

def detectar_alucinacion(texto):
    ## Devuelve los terminos morfologicos especificos hallados (no deberian estar)
    t = texto.lower()
    return [term for term in _TERMINOS_ALUCINACION if term in t]

## --- Tamano de la muestra a evaluar ---
N_EVAL = 30   ## ajustar segun cuanto se quiera evaluar (cada informe tarda unos segundos)

test_df_eval = pd.read_csv(test_path)
## Muestra estratificada: intentar cubrir todas las categorias BI-RADS
## Muestra estratificada por categoria BI-RADS (sin groupby.apply para evitar
## el DeprecationWarning de pandas). Se toma una porcion de cada categoria.
_por_categoria = max(1, N_EVAL // 5)
_partes = []
for _b, _grupo in test_df_eval.groupby("birads"):
    _n = min(len(_grupo), _por_categoria)
    _partes.append(_grupo.sample(_n, random_state=42))
muestra = pd.concat(_partes).head(N_EVAL).reset_index(drop=True)
print(f"Evaluando {len(muestra)} informes generados...")
print()

validator = CoherenceValidator(use_medical_vocabulary=True)
resultados = []

for i, row in muestra.iterrows():
    ## Generar informe (sin mostrar imagen, para no saturar la salida)
    res = generar_informe_para_imagen(row["image_path"], mostrar=False)
    informe = res["report"]
    birads_predicho = res["birads_level"]          ## 1-5

    ## 1. FIDELIDAD FACTUAL: el BI-RADS del texto coincide con la prediccion?
    birads_en_texto = extraer_birads_del_texto(informe)
    fidelidad = 1.0 if (birads_en_texto == birads_predicho) else 0.0

    ## 2. SEGURIDAD: idioma y alucinacion
    fuga_idioma = contiene_no_latino(informe)
    alucinaciones = detectar_alucinacion(informe)

    ## 3. COHERENCIA (CoherenceValidator)
    record = PredictionRecord(
        image_id=str(row.get("image_id", i)),
        birads_pred=birads_predicho,
        birads_confidence=res["birads_confidence"],
        findings_pred=[],
        explanation_text=informe,
        ground_truth_birads=int(row["birads"]),
    )
    coh = validator.validate_record(record)

    ## Adherencia terminologica AJUSTADA (solo terminos que el modelo produce)
    adh_ajustada = validator.lexicon.compute_terminology_adherence_pertinent(
        informe, birads_predicho
    ) if validator.lexicon else 0.0

    resultados.append({
        "birads_real": int(row["birads"]),
        "birads_predicho": birads_predicho,
        "birads_en_texto": birads_en_texto,
        "fidelidad_factual": fidelidad,
        "fuga_idioma": fuga_idioma,
        "n_alucinaciones": len(alucinaciones),
        "overall_coherence": coh["overall_coherence"],
        "terminology_adherence": coh["terminology_adherence"],
        "terminology_adherence_pertinent": adh_ajustada,
        "recommendation_coherence": coh["recommendation_coherence"],
        "explanation_completeness": coh["explanation_completeness"],
    })
    if (i + 1) % 5 == 0:
        print(f"  procesados {i+1}/{len(muestra)}")

df_eval = pd.DataFrame(resultados)

## --- Reporte agregado ---
print_section("EVALUACION DEL GENERADOR DE INFORMES (sin referencia)", width=70)
print(f"  Informes evaluados: {len(df_eval)}")
print()
print("  FIDELIDAD FACTUAL (BI-RADS del texto == prediccion):")
print(f"    Tasa de fidelidad: {df_eval['fidelidad_factual'].mean():.3f}")
print()
print("  SEGURIDAD:")
print(f"    Informes en espanol limpio (sin fuga): {(~df_eval['fuga_idioma']).mean():.3f}")
print(f"    Informes sin alucinaciones morfologicas: {(df_eval['n_alucinaciones']==0).mean():.3f}")
print()
print("  COHERENCIA CLINICA (CoherenceValidator):")
print(f"    Coherencia general (promedio): {df_eval['overall_coherence'].mean():.3f}")
print(f"    Adherencia terminologica (cruda):    {df_eval['terminology_adherence'].mean():.3f}")
print(f"    Adherencia terminologica (ajustada): {df_eval['terminology_adherence_pertinent'].mean():.3f}")
print(f"    Coherencia de recomendacion:   {df_eval['recommendation_coherence'].mean():.3f}")
print(f"    Completitud de la explicacion: {df_eval['explanation_completeness'].mean():.3f}")

## Guardar resultados
eval_path = RESULTS_DIR / f"evaluacion_informes_sin_referencia_{EXPERIMENT_FINAL}_{get_timestamp()}.json"
save_json({
    "experimento": EXPERIMENT_FINAL,
    "n_evaluados": len(df_eval),
    "fidelidad_factual": float(df_eval["fidelidad_factual"].mean()),
    "tasa_espanol_limpio": float((~df_eval["fuga_idioma"]).mean()),
    "tasa_sin_alucinacion": float((df_eval["n_alucinaciones"]==0).mean()),
    "coherencia_general": float(df_eval["overall_coherence"].mean()),
    "adherencia_terminologica_cruda": float(df_eval["terminology_adherence"].mean()),
    "adherencia_terminologica_ajustada": float(df_eval["terminology_adherence_pertinent"].mean()),
    "coherencia_recomendacion": float(df_eval["recommendation_coherence"].mean()),
    "detalle_por_informe": resultados,
}, eval_path)
print(f"\nResultados guardados: {eval_path.name}")